In [3]:
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf

In [4]:
SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "../models/best_model.pt"

In [5]:
class TinyKWS(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = self.gap(x).squeeze(-1).squeeze(-1)
        x = self.fc(x)
        return x

In [6]:
model = TinyKWS().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# =====================
# AUDIO LOADER
# =====================
def load_wav(path):
    wav, sr = sf.read(path)
    wav = torch.tensor(wav).float()

    # stereo → mono
    if len(wav.shape) > 1:
        wav = wav.mean(dim=1)

    wav = wav.unsqueeze(0)

    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    return wav

In [7]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=40,
    n_fft=400,
    hop_length=160
)

db_transform = torchaudio.transforms.AmplitudeToDB()

In [20]:
def predict(wav_path, threshold=0.5):
    wav = load_wav(wav_path)

    # force 1 sec
    if wav.shape[1] < SAMPLE_RATE:
        wav = F.pad(wav, (0, SAMPLE_RATE - wav.shape[1]))
    else:
        wav = wav[:, :SAMPLE_RATE]

    mel = mel_transform(wav)
    mel = db_transform(mel)
    mel = mel.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(mel)
        prob = torch.sigmoid(out).item()

    is_stop = prob > threshold

    if is_stop:
        print("Keyword detected : STOP")
        print("Alert Triggered ⚠️")
    else:
        print("No keyword detected")

In [ ]:
if __name__ == "__main__":
    predict("../samples/test-stop.wav")

Keyword detected : STOP
Alert Triggered ⚠️


In [22]:
if __name__ == "__main__":
    predict("../samples/test-step.wav")

No keyword detected
